In [1]:
import re 
import time

import pandas as pd 
import chromadb

In [2]:
df = pd.read_csv("data/Train.csv")
print(df.shape)
df.head()

(3081, 8)


,ID,text,Qatar Related,Geography,Politics & Conflict,Health & Wellbeing,Science,Sports
0,fL8IqbB5haFSy2gk,كشف مسؤول رياضي مصري أن الميدالية الذهبية التي...,1,G-7,NO_POLITICS,NO_HEALTH,NO_SCIENCE,SP-3
1,Jivkt5fbyLOBsaQd,أشاد وزير الدولة للشؤون الخارجية لدولة قطر، سل...,1,G-8,PC-1,NO_HEALTH,NO_SCIENCE,NO_SPORTS
2,3m9k8ipWupvzY9VG,بدأ نجوم السينما والتلفزيون وضيوف الحفل السنوي...,1,G-7,PC-1,NO_HEALTH,NO_SCIENCE,SP-2
3,kIuSWafG6YlQSoiA,ونقلت فرانس برس عن إنفانتينو قوله نحن في مرحلة...,0,G-2,NO_POLITICS,NO_HEALTH,NO_SCIENCE,SP-2
4,L1tA6JMVJ7fbMGKm,اعلنت وزارة الداخلية القطرية، الأربعاء، أن الت...,1,G-4,PC-1,NO_HEALTH,SC-3,NO_SPORTS


In [3]:
junk = df["text"].str.contains("urn:schemas", na=False)
print(junk.sum())
df[junk]["text"].str[:80].head()


0


Series([], Name: text, dtype: str)

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print(model.max_seq_length)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

128


In [5]:
print(model)

vec = model.encode("اهلا وسهلا")
print(vec.shape)
print(vec[:5])

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
)
(384,)
[ 0.12244345  0.03684181  0.24197623  0.06526392 -0.04337892]


In [6]:
from sentence_transformers import util

a = model.encode("أنا سعيد جداً")
b = model.encode("أشعر بالفرح")
c = model.encode("السيارة معطلة")

print(util.cos_sim(a, b))
print(util.cos_sim(a, c))

tensor([[0.9252]])
tensor([[-0.0375]])


In [7]:
tokenizer = model.tokenizer
lengths = df["text"].apply(lambda t: len(tokenizer.tokenize(t)))
print(lengths.describe())
print((lengths > model.max_seq_length).mean())

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (444 > 128). Running this sequence through the model will result in indexing errors


count    3081.000000
mean      423.959104
std       326.442733
min        26.000000
25%       242.000000
50%       343.000000
75%       504.000000
max      5015.000000
Name: text, dtype: float64
0.9500162284972412


In [8]:
MAX_TOKENS = 120

def chunk_text(text, max_tokens=MAX_TOKENS):
    units = [u.strip() for u in re.split(r'(?<=[.!?؟])\s+|\n+', text) if u.strip()]
    chunks, current, current_len = [], [], 0
    for u in units:
        n = len(tokenizer.tokenize(u))
        if current and current_len + n > max_tokens:
            chunks.append(" ".join(current))
            current, current_len = [], 0
        current.append(u)
        current_len += n
    if current:
        chunks.append(" ".join(current))
    return chunks

rows = []
for _, r in df.iterrows():
    for i, c in enumerate(chunk_text(r["text"])):
        rows.append({"chunk_id": f'{r["ID"]}_{i}', "parent_id": r["ID"], "chunk_text": c})

chunks_df = pd.DataFrame(rows)
print(len(df), len(chunks_df))
print(chunks_df["chunk_id"].is_unique)

too_long = chunks_df["chunk_text"].apply(lambda t: len(tokenizer.tokenize(t)) > MAX_TOKENS)
print(too_long.sum())

3081 14431
True
183


In [9]:
texts = chunks_df["chunk_text"].tolist()

start = time.time()
embeddings = model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
print(time.time() - start)
print(embeddings.shape)

Batches:   0%|          | 0/226 [00:00<?, ?it/s]

526.7514133453369
(14431, 384)


In [10]:
client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_or_create_collection(name="arabic_news", metadata={"hnsw:space": "cosine"})

BATCH = 500
for i in range(0, len(chunks_df), BATCH):
    part = chunks_df.iloc[i:i + BATCH]
    collection.upsert(
        ids=part["chunk_id"].tolist(),
        embeddings=embeddings[i:i + BATCH].tolist(),
        documents=part["chunk_text"].tolist(),
        metadatas=[{"parent_id": p} for p in part["parent_id"]],
    )

print(collection.count())

14431


In [11]:
def search(query, n_results=3):
    query_vec = model.encode([query], normalize_embeddings=True).tolist()
    return collection.query(query_embeddings=query_vec, n_results=n_results)

def show(query, n_results=3):
    res = search(query, n_results)
    for doc, dist, meta in zip(res["documents"][0], res["distances"][0], res["metadatas"][0]):
        print(f"similarity={1 - dist:.3f} | parent={meta['parent_id']}")
        print(doc[:200], "\n")

show("كرة قدم")

similarity=0.621 | parent=nYCsXoblu17rNy8H
مثل باقي الفرق الكبرى، قرر نادي برشلونة أن كرة القدم الآن هي "لعبة أرقام"، حسب ما كتبه كريس أندرسن وديفيد سالّي في كتابهما الذي يحمل نفس الاسم. ويبحث مديرو ومدربو الفرق عن فائدة من احصائيات اللعبة: من 

similarity=0.620 | parent=OYvSwZj24AQ079lU
وقال الرائد الروسي قبل انطلاقه من الأرض سنأخذ معنا هذه الكرة، ربما تكون هي الكرة المستخدمة في المباراة الأولى وسيبقى الرواد في المحطة أشهر، لكن الكرة ستعود إلى الأرض بعد أقل من أسبوعين مع الرائد الروس 

similarity=0.618 | parent=nYCsXoblu17rNy8H
لم تجد شركة تليفونيكا أفضل من فريق برشلونة لكي تعمل معه. فالفريق مشهور بتطويره للعبة كرة القدم... ليس بسبب المهارات المبهرة والبارعة للاعبيه البارزين من أمثال ليونيل ميسّي، ولكن بسبب النمط الجديد الذي 



In [12]:
def label_hits(query, column, no_value, k=3, fetch=15):
    res = search(query, n_results=fetch)
    parents = list(dict.fromkeys(m["parent_id"] for m in res["metadatas"][0]))[:k]
    labels = df.set_index("ID").loc[parents, column]
    return int((labels != no_value).sum())

print(label_hits("كرة القدم", "Sports", "NO_SPORTS"))
print(label_hits("الصحة والأمراض والمستشفيات", "Health & Wellbeing", "NO_HEALTH"))
print(label_hits("الحرب والصراع السياسي", "Politics & Conflict", "NO_POLITICS"))

3
3
3


In [13]:
show("اتفاق بين قطر وألمانيا")
show("اكتشاف كوكب جديد")

similarity=0.827 | parent=GvhBIS7g9vUpcYNZ
أعلنت شركة "بورش" الألمانية لصناعة السيارات يوم 14 اغسطس/آب عن التوصل إلى اتفاق حول دخول دولة قطر كمساهم من خلال شراء 10% من الأسهم الأصلية لشركة بورش القابضة. وينص الاتفاق على حصول قطر على حصة كبيرة  

similarity=0.801 | parent=Ehv1zJJ3WDSLVFCB
قطر بدأت التفاوض مع ألمانيا لشراء حصة في مجموعة "إي أي دي أس" (الأوروبية) كشفت مصادر حكومية ببرلين لوكالة الأنباء الألمانية أن قطر تسعى لشراء جزء من حصة شركة دايملر الألمانية للسيارات في المجموعة الأو 

similarity=0.775 | parent=Yh9ThBhomIz3ZNhf
شهدت فيينا التوقيع على مذكرتي تفاهم بين قطر والنمسا خلال زيارة أمير دولة قطر الشيخ تميم بن حمد آل ثاني للنمسا، بهدف تعزيز العلاقات الاقتصادية والتجارية والإعفاء من بعض الرسوم الجمركية بين البلدين. وبح 

similarity=0.666 | parent=XjejmLP0cRXGY8Uf
اكتشف علماء الفلك بمساعدة تلسكوب هابل كويكبا مزدوجا فريدا من نوعه، حسبما نشر موقع Space.com. وحدد العلماء موقع الكويكب بين المريخ والمشتري ويتكون من جزئين، ويعتبر فريدا من نوعه كونه مزدوجا. ويصنفه بعض 

similarity=0.651